# ARK-018 V4 — Resume-only launcher after Colab disconnect

Use **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

This launcher does **not** start a new experiment and does not alter the preregistration. It resumes the existing Drive-backed ARK-018 campaign from the last exact checkpoint for each unfinished arm, while completed arms are reused.

Scientific execution is frozen at `fb0420b7a46521a5f14d125564078ca1c6336d78`.

Drive input: `/content/drive/MyDrive/genisis-arkenstone/data_15.parquet`

Drive state: `/content/drive/MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1/`


In [ ]:
%pip -q install 'pyarrow==21.0.0' 'tokenizers==0.21.4' 'datasets==4.0.0'
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU required: Runtime → Change runtime type → T4 GPU')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
SCIENCE = Path('/content/drive/MyDrive/genisis-arkenstone/data_15.parquet')
ROOT = Path('/content/drive/MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1')
if not SCIENCE.exists():
    raise FileNotFoundError(f'Missing frozen science file: {SCIENCE}')
if not ROOT.exists():
    raise FileNotFoundError(f'Missing existing ARK-018 state: {ROOT}')
print('Science:', SCIENCE, SCIENCE.stat().st_size, 'bytes')
print('Resume root:', ROOT)


In [ ]:
# Inspect the ACTUAL Drive state before execution. Partial JSON can be newer than the exact model checkpoint.
import json, torch
SEEDS = [31801, 31902]
ARMS = ['SCIENCE_ONLY','BIRTH_NATURAL_2PCT','BIRTH_REHEARSAL_10PCT','SCIENCE_REPLAY_10PCT_CONTROL']
prepared_path = ROOT/'prepared'/'ARK-018_PREPARED_RECEIPT.json'
if not prepared_path.exists():
    raise RuntimeError('Prepared Drive cache is missing. Use the full V4 launcher instead of resume-only.')
prepared = json.loads(prepared_path.read_text())
horizon = int(prepared['horizon_updates'])
print('Frozen horizon:', horizon, 'updates per arm')
print('\nDRIVE RESUME TABLE')
print(f"{'seed':>7}  {'arm':35} {'partial':>8} {'checkpoint':>10}  action")
print('-'*82)
remaining = 0
for seed in SEEDS:
    for arm in ARMS:
        pp = ROOT/'results'/f'ARK-018_SEED_{seed}_{arm}_PARTIAL.json'
        cp = ROOT/'checkpoints'/f'seed_{seed}'/f'{arm}.pt'
        partial_step = None
        if pp.exists():
            try: partial_step = int(json.loads(pp.read_text()).get('step', -1))
            except Exception: partial_step = -1
        ckpt_step = None
        if cp.exists():
            obj = torch.load(cp, map_location='cpu', weights_only=False)
            ckpt_step = int(obj.get('step', -1))
            del obj
        if ckpt_step == horizon:
            action = 'REUSE COMPLETE'
        elif ckpt_step is not None and ckpt_step >= 0:
            action = f'RESUME FROM {ckpt_step}'
            remaining += horizon - ckpt_step
        else:
            action = 'START UNTOUCHED ARM'
            remaining += horizon
        print(f'{seed:7d}  {arm:35} {str(partial_step):>8} {str(ckpt_step):>10}  {action}')
print('\nEstimated remaining optimizer updates from exact checkpoints:', remaining)
print('Note: if partial=6500 but checkpoint=6000, steps 6001–6500 are intentionally replayed from the exact checkpoint.')


In [ ]:
# Fetch the exact frozen scientific runner. The live branch may move; execution may not.
import os, shutil, subprocess
REPO = '/content/An-Ra-the-new-AGI'
FROZEN = 'fb0420b7a46521a5f14d125564078ca1c6336d78'
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','-q',FROZEN], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == FROZEN, (head, FROZEN)
print('Frozen execution checkout:', head)


In [ ]:
# Static compile gate before touching training state.
import subprocess
files = [
    'experiments/ARK-018/ark018_v3_common.py',
    'experiments/ARK-018/run_ark018_science_birth_v3.py',
    'experiments/ARK-018/ark018_v3_binding_fast.py',
    'experiments/ARK-018/run_ark018_science_birth_v4.py',
]
subprocess.run(['python','-m','py_compile', *[f'{REPO}/{x}' for x in files]], check=True)
print('Compile gate PASS')


In [ ]:
# Resume the SAME campaign. No --force-prepare. The frozen runner skips completed arms and resumes incomplete ones.
cmd = [
    'python', f'{REPO}/experiments/ARK-018/run_ark018_science_birth_v4.py',
    '--mode','all',
    '--enable-sciq',
    '--science-path', str(SCIENCE),
    '--expected-head', FROZEN,
]
print('Starting resume...')
subprocess.run(cmd, cwd=REPO, check=True)


In [ ]:
# Final artifact check. The authoritative copy remains in Drive.
FINAL = ROOT/'results'/'ARKENSTONE_ARK018_SCIENCE_BIRTH_RESULTS.zip'
RESULT = ROOT/'results'/'ARK-018_RESULT.json'
if RESULT.exists():
    r = json.loads(RESULT.read_text())
    print('Campaign status:', r.get('status'), '| verdict:', r.get('primary_verdict'))
if FINAL.exists():
    import hashlib
    h = hashlib.sha256()
    with FINAL.open('rb') as f:
        for b in iter(lambda: f.read(8<<20), b''): h.update(b)
    print('FINAL ZIP:', FINAL)
    print('bytes:', FINAL.stat().st_size, '| sha256:', h.hexdigest())
else:
    print('Final ZIP not present yet; the run did not reach campaign completion.')
